# CS 483/583 · Module 10 · Self-Supervised Learning and Embeddings

A drawer holds thousands of sensor records but nobody labeled which ones contain a change of interest. Can the record itself pose a question whose answer we already know? Hide one reading, or make two views of the same record, and use the result to teach an encoder.

Module 09 converted raw readings into hidden features such as the strongest rise and fall. Its calibrated detector needed reference measurements. Here we choose tasks that get their training targets from the recorded signals themselves.

## Running the examples

Use a Python 3 notebook with PyTorch installed. A CPU is sufficient. Run the cells in order, then change an input and predict what will happen.

Can the readings [2, 4, 6] supply an input and a target for training without a person labeling the trend?

<details><summary>Reveal answer</summary>

Yes. Give the first two readings to the model and save 6 as the target. Keeping 6 hidden matters: if it appears in the input, copying it would solve the task.

</details>

## Make a target from the record

A pretext task is a practice task chosen to teach features. In self-supervised learning, the training target comes from the data or from a known transformation of the data. Start by hiding the last reading. Split whole independent records into training and evaluation groups before creating examples.

For our fictional three-second records the next reading often continues a short trend. Use [2, 4] to predict 6. A simple baseline repeats the last seen reading and predicts 4. Its squared error is 4. A trend predictor can do better on this particular record.

**Formal notation.** Write the full record as (x₁, x₂, x₃). The model receives (x₁, x₂). The loss (prediction − x₃)² can be computed after making the prediction.

**Predict before you run.** A classmate creates two slightly different copies of each record, then randomly splits all copies. What might go wrong?

In [1]:
record = [2., 4., 6.]
first, second = record[:2]
target = record[2]
for guess in (second, second + (second - first)):
    print("prediction", guess, "squared error", (guess-target)**2)

prediction 4.0 squared error 4.0
prediction 6.0 squared error 0.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Nearly identical versions could occur in training and testing. Split by original record first and keep its derived views together.

</details>

## Encode the visible part and reconstruct the hidden part

An encoder converts the visible readings to a feature, also called an embedding. Our encoder produces one number from two. A decoder uses that number to predict the missing reading. We fit the encoder by reducing reconstruction error on four fictional training records.

Let the decoder add the feature to the second reading. Then a useful feature is the expected change from second to third. Fitting the two encoder weights is the least-squares regression from Module 05, with the next recorded change as its target.

**Formal notation.** For visible readings a,b, embedding z = w₁a + w₂b. The reconstruction is b + z. Fit w to minimize the sum of (b + z − c)², where c is the hidden reading. That is the same as fitting z to c − b.

**Predict before you run.** Why does the fit give approximately [−1, 1] on these records?

In [2]:
import torch
records = torch.tensor([[1.,2.,3.],[3.,2.,1.],[1.,3.,5.],[5.,4.,3.]])
visible = records[:, :2]
target_change = records[:, 2] - records[:, 1]
weights = torch.linalg.lstsq(visible, target_change).solution
new = torch.tensor([4., 7.])
feature = new @ weights
print("weights", [round(w, 2) for w in weights.tolist()])
print("embedding", round(feature.item(), 2))
print("reconstruction", round((new[1]+feature).item(), 2))

weights [-1.0, 1.0]
embedding 3.0
reconstruction 10.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

In every example c − b equals b − a. We can compute that change with −1 times the first reading and +1 times the second. The training inputs include independent directions, so these weights are identified here.

</details>

## Ask what the embedding preserves

The fitted encoder keeps the change across the first two readings. It discards their absolute brightness. An embedding is useful relative to a task: keeping change may help distinguish rising and falling signals, while a question about absolute brightness needs other features.

Both [1, 3] and [7, 9] become 2. The decoder still has access to the second reading and can predict different final values. A later classifier given only the embedding cannot recover the absolute level.

**Formal notation.** If z(a,b) = b − a, then z(a + k,b + k) = (b + k) − (a + k) = z(a,b). Adding the same offset k leaves this feature unchanged.

**Predict before you run.** Can a predictor using only z determine whether the second reading exceeds 8?

In [3]:
for a, b in [(1,3), (7,9), (7,5)]:
    z = b-a
    print("visible", (a,b), "embedding", z, "decoded last reading", b+z)

visible (1, 3) embedding 2 decoded last reading 5
visible (7, 9) embedding 2 decoded last reading 11
visible (7, 5) embedding -2 decoded last reading 3


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

No. The first two examples have z = 2, but only [7, 9] passes that threshold. Keeping the second reading would resolve the ambiguity.

</details>

## Compare two embeddings by their direction

A different encoder might keep several features. For example, give a record one coordinate for strongest rise and one for strongest fall. The dot product multiplies matching coordinates and adds. Cosine similarity divides that product by the length of each vector, allowing comparison of direction despite differences in size.

Take query [2, 0]. Candidate A is [1, 0], which points the same way. Candidate B is [0, 5], which points at a right angle. Candidate C is [3, 4], length 5. Its cosine with the query is 6/(2 × 5) = 0.6.

**Formal notation.** For nonzero vectors u,v, cos(u,v) = (u · v)/(‖u‖ ‖v‖). For [p,q], length ‖u‖ is the square root of p² + q². A cosine of 1 means equal direction, even for unequal magnitudes.

**Predict before you run.** Why might cosine be a poor choice if absolute jump size determines the class?

In [4]:
import torch
from torch.nn.functional import cosine_similarity
query = torch.tensor([[2., 0.]])
candidates = torch.tensor([[1.,0.],[0.,5.],[3.,4.]])
print([round(v, 2) for v in cosine_similarity(query, candidates).tolist()])

[1.0, 0.0, 0.6]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

[1, 0] and [10, 0] get cosine 1 with the query. A feature or distance that retains magnitude would make the jump-size difference available.

</details>

## Make positive and negative pairs

In contrastive learning an anchor is a reference example. A positive is a view it should match. Negatives are alternatives. A useful augmentation changes something you expect to be irrelevant to the downstream question. That expectation is a modeling assumption you can test.

For trend direction, adding 5 to every reading preserves differences. Treat [1, 2, 3] and [6, 7, 8] as a positive pair from the same source. Reversing the order gives a falling signal, so using reversal as a positive would teach the wrong invariance for this task.

**Formal notation.** An encoder f maps each view x to z = f(x). The pair-selection rule supplies the match label. Here trend(t) = last(t) − first(t) is a simple feature to inspect the transformation.

**Predict before you run.** Suppose a different rising record is treated as a negative. Why might that matter?

In [5]:
record = [1, 2, 3]
views = [record, [v+5 for v in record], list(reversed(record))]
for view in views:
    print(view, "net change", view[-1]-view[0])

[1, 2, 3] net change 2
[6, 7, 8] net change 2
[3, 2, 1] net change -2


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The two records have distinct identities, but share the downstream trend class. Pushing them apart may suppress useful shared structure. Such a negative is called a false negative relative to that downstream meaning.

</details>

## Write a loss that rewards the right match

Give each candidate a score for similarity to the anchor. Convert scores into positive weights with the exponential function, then divide by their sum. The resulting match probabilities add to one. Penalize a small probability for the positive with its negative natural logarithm, as in classification.

Use two candidates, positive score 1 and negative score 0, at temperature 1. The positive gets e/(e + 1), about 0.731, and loss about 0.313. If the scores tie, each gets 0.5 and loss rises to about 0.693.

**Formal notation.** With positive score s₊, candidate scores sⱼ, and temperature τ > 0, p₊ = exp(s₊/τ) / Σⱼ exp(sⱼ/τ). The sum includes the positive. For this one anchor the loss is −log(p₊).

**Predict before you run.** If a third alternative has score 0, what happens to the first example’s loss?

In [6]:
import torch
from torch.nn.functional import cross_entropy
scores = torch.tensor([[1., 0.], [0., 0.]])
positive_index = torch.tensor([0, 0])
print("positive probabilities", [round(x,3) for x in scores.softmax(dim=1)[:,0].tolist()])
print("losses", [round(x,3) for x in cross_entropy(scores, positive_index, reduction="none").tolist()])

positive probabilities [0.731, 0.5]
losses [0.313, 0.693]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The numerator remains e. The denominator increases from e + 1 to e + 2. The positive probability falls from about 0.731 to 0.576, so the loss rises to about 0.551.

</details>

## See a contrastive objective change a weight

To distinguish rising from falling, try a feature with a fixed first coordinate 1 and a learned second coordinate equal to gain times signed change. Increasing the gain makes opposite trends point farther apart. Train the gain using one known positive pair and one negative.

The anchor rises by 1. A brightness-shifted copy also rises by 1. The negative falls by 1. Initially gain is 0.5, giving [1, 0.5] for the anchor and [1, −0.5] for the negative. Cosine with the positive is 1, while cosine with the negative starts at 0.6.

**Formal notation.** f(d) = [1, g·d] for signed change d and learned gain g. Apply the two-candidate loss to the two cosines, then use the Module 03 gradient update to adjust g.

**Predict before you run.** Does lowering this loss on one pair show that the encoder can identify every kind of trend?

In [7]:
import torch
import torch.nn.functional as F
gain = torch.tensor(0.5, requires_grad=True)
def embed(change):
    return torch.stack([torch.ones_like(gain), gain * change])
for step in range(6):
    anchor = embed(1.)
    candidates = torch.stack([embed(1.), embed(-1.)])
    scores = F.cosine_similarity(anchor[None, :], candidates)
    loss = F.cross_entropy(scores[None, :], torch.tensor([0]))
    if step in (0,5):
        print(step, "gain", round(gain.item(),3), "loss", round(loss.item(),3))
    if step < 5:
        gradient, = torch.autograd.grad(loss, gain)
        gain = (gain - 0.25*gradient).detach().requires_grad_()

0 gain 0.5 loss 0.513
5 gain 1.009 loss 0.311


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

It shows that the update improves these supplied matches. Test the representation on independent records, including cases unlike this simple pair.

</details>

## Freeze the encoder and test it on a new task

Pretraining fits an encoder using a practice objective. For a downstream task, keep its weights fixed and train a simple classifier, called a probe, on labeled examples. The test labels stay out of both fits. Here we ask whether a record is rising, using new synthetic records. Compare the probe with a constant prediction and a simple rule that directly checks whether the second reading exceeds the first.

Use the reconstruction encoder from earlier. In the labeled probe-training set its rising embeddings are 2 and 1, the falling embeddings −3 and −2. Put a threshold halfway between the two class means. Use that threshold on four different labeled records and inspect each prediction.

**Formal notation.** The class means are 1.5 and −2.5, so threshold t = (1.5 − 2.5)/2 = −0.5. Predict rising if z > t. This is a one-dimensional linear decision rule with score z − t.

**Predict before you run.** What do four correct predictions by both the trained probe and the raw-change rule tell us? Which benefit of pretraining would need another experiment?

In [8]:
import torch
pre = torch.tensor([[1.,2.,3.],[3.,2.,1.],[1.,3.,5.],[5.,4.,3.]])
weights = torch.linalg.lstsq(pre[:,:2], pre[:,2]-pre[:,1]).solution
train = torch.tensor([[4.,6.],[9.,6.],[3.,4.],[7.,5.]])
labels = torch.tensor([1,0,1,0])
z = train @ weights
threshold = (z[labels==1].mean() + z[labels==0].mean())/2
test = torch.tensor([[2.,5.],[7.,6.],[3.,5.],[9.,7.]])
truth = torch.tensor([1,0,1,0])
pred = (test @ weights > threshold).long()
print("threshold", round(threshold.item(),2))
print("predictions", pred.tolist(), "correct", int((pred==truth).sum()), "of", len(truth))
# Two comparisons on exactly the same four test records.
majority_guess = torch.zeros_like(truth)  # class 0 breaks the training-count tie
raw_change = (test[:, 1] > test[:, 0]).long()
for name, guesses in [("constant", majority_guess), ("raw change", raw_change)]:
    print(name, "correct", int((guesses == truth).sum()), "of", len(truth))

threshold -0.5
predictions [1, 0, 1, 0] correct 4 of 4
constant correct 2 of 4
raw change correct 4 of 4


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The frozen embedding retains enough information for the probe to recognize these trends. The raw-change rule also gets four of four, so this comparison demonstrates no accuracy gain over that rule. Try noisier records from unseen instruments and evaluate the same frozen predictors, including the baseline, on the same examples.

</details>

## Exercise 1

Design a masked prediction task for [4, 7, 10]. Name the visible input, hidden target and one way to leak the answer.

<details><summary>Hint</summary>

The pretext task needs a target you can generate from a recorded example.

</details>

Check your work.

<details><summary>Reveal answer</summary>

For last-reading prediction, show [4, 7] and hide 10. Passing the full record, or giving the mean of all three readings alongside the first two, would reveal the target.

</details>

## Exercise 2

Use z = b − a to encode [5, 7] and [1, 3]. Which downstream question could distinguish these inputs: trend direction or second reading above 5?

<details><summary>Hint</summary>

List the two z values before considering each question.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Both embeddings are 2, and both records rise. The second readings lie on opposite sides of 5. The embedding cannot distinguish that fact. Retaining b alongside z would allow both tasks.

</details>

## Exercise 3

Rank [0, 2], [4, 3] and [5, 0] by cosine with [1, 0]. Give the three cosines.

<details><summary>Hint</summary>

The dot product with [1,0] selects the first coordinate. Divide by candidate length.

</details>

Check your work.

<details><summary>Reveal answer</summary>

[5,0] scores 1, [4,3] scores 4/5 = 0.8, and [0,2] scores 0. Cosine measures angle here.

</details>

## Exercise 4

One anchor has three candidates. The positive has score 0, while the others have scores 0 and ln(2). At temperature 1, find the positive probability and loss.

<details><summary>Hint</summary>

Exponentiating those scores gives weights 1, 1 and 2.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The total is 4. The positive receives 1/4, so the loss is −ln(1/4) = ln(4), about 1.386. The most similar alternative gets half of the probability.

</details>

## Exercise 5

The model should retrieve the same written digit across photographs. Two distinct photographs of the same digit are used as negatives. What assumption caused the problem, and how might you investigate it?

<details><summary>Hint</summary>

Compare the source identity used to create pairs with the desired downstream meaning.

</details>

Check your work.

<details><summary>Reveal answer</summary>

A different source image was assumed to mean a different semantic object or class. Inspect difficult negative pairs with available digit labels, then compare retrieval on an independent labeled set under alternative pair-selection rules.

</details>

## Exercise 6

Probe-training class means are −1 and +3. Use the midpoint rule on test embeddings 0, 1.5 and 4 with labels falling, rising and falling. Report accuracy and name the error.

<details><summary>Hint</summary>

Use a strict greater-than comparison with the midpoint.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The threshold is 1. Predictions are falling, rising, rising. Two of three are correct. The final example is a false rising prediction. Its original record would help diagnose why its embedding falls on that side.

</details>

We compared one embedding with several candidates, then normalized the scores to distribute weight across them. Module 11 applies that calculation inside a sequence. Each position can compare itself with earlier positions and mix information from them.

If a query gives two earlier readings equal scores and both are allowed, how much weight will each get after exponential normalization?

<details><summary>Reveal answer</summary>

Equal scores give equal positive weights. With only those two candidates, dividing by their sum gives one half to each. The next module uses those weights to combine the values at the two positions.

</details>